# 07 · Lista de casa: PyTorch, fundamentos

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

## Instruções

- Trabalhe em uma cópia deste notebook e substitua cada `...` pelo seu código.
- Não use a função `treinar` da aula: na questão 3 o laço é seu.
- Cerca de metade da nota está nas respostas escritas. Use os números que você obteve.
- Antes de entregar, use **Reiniciar sessão e executar tudo**.

| Questão | Tema | Pontos |
|---|---|---|
| 1 | Autograd e derivada numérica | 2,0 |
| 2 | Taxa de aprendizado | 2,0 |
| 3 | Laço de treino completo para "nota 5" | 3,0 |
| 4 | Caça aos erros | 2,0 |
| 5 | PyTorch ou scikit-learn? | 1,0 |

In [ ]:
import copy
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.datasets import make_moons
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch import nn

SEMENTE = 42
torch.manual_seed(SEMENTE)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "· dispositivo:", dispositivo)

## Questão 1 (2,0 pontos): autograd e derivada numérica

Uma forma independente de conferir um gradiente é a **derivada numérica**: mexer um pouco em um parâmetro e ver quanto a função muda.

`derivada em relação a w[i] ≈ ( f(w + h·eᵢ) − f(w − h·eᵢ) ) / (2h)`

em que `eᵢ` é um vetor de zeros com 1 na posição `i`. Nesta questão os tensores estão em `float64`, para que a conta numérica tenha precisão.

a) Calcule o gradiente de `f` em `w_q1` com o autograd e guarde em `grad_q1` (um tensor de 4 números, sem histórico).

b) Complete `derivada_numerica(f, w, i, h)` e monte `grad_num_q1`, um tensor com as 4 derivadas numéricas.

In [ ]:
torch.manual_seed(5)
X_q1 = torch.randn(40, 4, dtype=torch.float64)
y_q1 = torch.randn(40, dtype=torch.float64)

def f(w):
    return (torch.tanh(X_q1 @ w) - y_q1).pow(2).mean()

w_q1 = torch.tensor([0.3, -0.7, 1.2, 0.1], dtype=torch.float64, requires_grad=True)

In [ ]:
# ✏️ Questão 1
# SEU CÓDIGO AQUI
grad_q1 = ...


def derivada_numerica(f, w, i, h=1e-5):
    # SEU CÓDIGO AQUI
    ...


# SEU CÓDIGO AQUI
grad_num_q1 = ...

In [ ]:
# ✅ Conferência da questão 1 (não altere)
assert grad_q1.shape == (4,) and grad_num_q1.shape == (4,)
assert grad_q1.abs().sum() > 0.1, "o gradiente está zerado"
assert torch.allclose(grad_q1.detach(), grad_num_q1.detach(), atol=1e-6), "autograd e derivada numérica não coincidem (se executou a célula duas vezes, o gradiente acumulou: recrie w_q1)"
print("Os dois gradientes coincidem. Maior diferença:", (grad_q1.detach() - grad_num_q1.detach()).abs().max().item())

**1c)** Quantas vezes a função `f` foi avaliada para obter o gradiente numérico? E com o autograd? O que isso implica para um modelo com 10 milhões de parâmetros? Cite ainda uma situação em que a derivada numérica continua útil.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 2 (2,0 pontos): taxa de aprendizado

a) Complete `treinar_reta(taxa, passos)`: descida do gradiente **à mão** (sem `nn` nem `optim`) para o modelo `w·x + b` nos dados abaixo, com erro quadrático médio, `w` e `b` começando em zero. A função devolve a lista com a perda de cada passo.

b) Monte `curvas_q2`, um dicionário que leva cada taxa de `TAXAS` à lista de perdas correspondente, e execute a célula do gráfico.

In [ ]:
torch.manual_seed(SEMENTE)
x_q2 = torch.linspace(-1, 1, 200).unsqueeze(1)
y_q2 = 3 * x_q2 + 0.5 + 0.2 * torch.randn(200, 1)
TAXAS = [0.01, 0.1, 0.5, 0.9, 1.1]

In [ ]:
# ✏️ Questão 2
def treinar_reta(taxa, passos=50):
    # SEU CÓDIGO AQUI
    ...
    return perdas


# SEU CÓDIGO AQUI
curvas_q2 = ...

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for taxa, perdas in curvas_q2.items():
    ax.plot(perdas, label=f"taxa {taxa}")
ax.set_yscale("log"); ax.set_xlabel("passo"); ax.set_ylabel("perda (escala logarítmica)"); ax.legend(frameon=False, fontsize=8)
print({taxa: float(f"{perdas[-1]:.3g}") for taxa, perdas in curvas_q2.items()})

In [ ]:
# ✅ Conferência da questão 2 (não altere)
assert list(curvas_q2) == TAXAS and all(len(p) == 50 for p in curvas_q2.values())
assert abs(curvas_q2[0.01][0] - curvas_q2[0.5][0]) < 1e-6, "a primeira perda (antes de qualquer atualização) deve ser a mesma para todas as taxas"
assert curvas_q2[0.5][-1] < 0.06, "com taxa 0,5 a perda deveria chegar perto de 0,04"
assert curvas_q2[0.01][-1] > curvas_q2[0.1][-1] > curvas_q2[0.5][-1] - 1e-9
assert curvas_q2[1.1][-1] > curvas_q2[1.1][0], "com taxa 1,1 a perda deveria crescer"
print("Formato correto.")

**2c)** Descreva o comportamento de cada uma das cinco curvas (lenta, adequada, oscilante, divergente). Qual taxa você escolheria e por quê? A perda mínima alcançada fica perto de que valor, e de onde vem esse valor?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 3 (3,0 pontos): laço de treino completo para "nota 5"

A célula abaixo prepara os dados: presença das 1000 palavras mais frequentes, alvo "a avaliação recebeu nota 5", e uma regressão logística do scikit-learn como referência.

In [ ]:
URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

df = pd.read_csv(ARQUIVO, dtype={"product_id": str}, low_memory=False)
textos = df["review_title"].fillna("") + ". " + df["review_text"].fillna("")
alvo = (df["overall_rating"] == 5).astype(int)

t_treino, t_teste, y_treino, y_teste = train_test_split(textos, alvo, test_size=0.2, stratify=alvo, random_state=SEMENTE)
t_ajuste, t_valid, y_ajuste, y_valid = train_test_split(t_treino, y_treino, test_size=0.25, stratify=y_treino, random_state=SEMENTE)

vetorizador = CountVectorizer(max_features=1000, binary=True, dtype=np.float32)
A, V = vetorizador.fit_transform(t_ajuste), vetorizador.transform(t_valid)
auc_referencia = roc_auc_score(y_valid, LogisticRegression(max_iter=1000).fit(A, y_ajuste).predict_proba(V)[:, 1])

X_aj = torch.from_numpy(A.toarray()).to(dispositivo)
X_va = torch.from_numpy(V.toarray()).to(dispositivo)
y_aj = torch.tensor(y_ajuste.to_numpy(), dtype=torch.float32, device=dispositivo)
y_va = torch.tensor(y_valid.to_numpy(), dtype=torch.float32, device=dispositivo)
print(f"ajuste {tuple(X_aj.shape)} · validação {tuple(X_va.shape)} · regressão logística (scikit-learn): AUC {auc_referencia:.4f}")

a) Escreva `treinar_q3(modelo, epocas, taxa=1e-3, lote=256)`. A função deve:

- treinar com mini-lotes embaralhados a cada época, `nn.BCEWithLogitsLoss` e `torch.optim.Adam`;
- ao fim de cada época, medir na validação a perda e a AUC (em modo de avaliação, sem gradientes);
- **guardar uma cópia dos parâmetros da época de menor perda de validação** (`copy.deepcopy(modelo.state_dict())`);
- devolver dois objetos: um DataFrame com as colunas `epoca`, `perda_ajuste`, `perda_valid` e `auc_valid`, e o `state_dict` guardado.

b) Crie `modelo_q3`, uma rede com pelo menos uma camada oculta, e treine por 12 épocas: `historico_q3, melhor_estado_q3 = treinar_q3(modelo_q3, 12)`.

c) Guarde em `melhor_epoca_q3` a época de menor perda de validação, carregue `melhor_estado_q3` em `modelo_q3` e execute a célula do gráfico.

In [ ]:
# ✏️ Questão 3
def treinar_q3(modelo, epocas, taxa=1e-3, lote=256):
    # SEU CÓDIGO AQUI
    ...
    return historico, melhor_estado


torch.manual_seed(SEMENTE)
# SEU CÓDIGO AQUI
modelo_q3 = ...
historico_q3, melhor_estado_q3 = ...
melhor_epoca_q3 = ...

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(historico_q3["epoca"], historico_q3["perda_ajuste"], color="#6b6b6b", label="ajuste")
ax.plot(historico_q3["epoca"], historico_q3["perda_valid"], color="#eb6834", label="validação")
ax.axvline(melhor_epoca_q3, color="#eb6834", ls=":", lw=1)
ax.set_xlabel("época"); ax.set_ylabel("perda"); ax.legend(frameon=False)
print(f"melhor época: {melhor_epoca_q3} · AUC nessa época: {historico_q3.set_index('epoca').loc[melhor_epoca_q3, 'auc_valid']:.4f} · AUC na última: {historico_q3['auc_valid'].iloc[-1]:.4f} · referência: {auc_referencia:.4f}")

In [ ]:
# ✅ Conferência da questão 3 (não altere)
assert list(historico_q3.columns) == ["epoca", "perda_ajuste", "perda_valid", "auc_valid"] and len(historico_q3) == 12
assert sum(p.numel() for p in modelo_q3.parameters()) > 1001, "o modelo deve ter pelo menos uma camada oculta"
assert historico_q3["perda_ajuste"].iloc[-1] < historico_q3["perda_ajuste"].iloc[0], "a perda de ajuste deveria cair ao longo do treino"
assert melhor_epoca_q3 == int(historico_q3.loc[historico_q3["perda_valid"].idxmin(), "epoca"])
modelo_q3.eval()
with torch.no_grad():
    _logits = modelo_q3(X_va).reshape(-1)
_perda_carregada = nn.BCEWithLogitsLoss()(_logits, y_va).item()
assert abs(_perda_carregada - historico_q3["perda_valid"].min()) < 1e-4, "o modelo carregado não reproduz a perda da melhor época: o estado foi copiado com deepcopy?"
assert historico_q3["auc_valid"].max() > auc_referencia - 0.01
print(f"Formato correto. O modelo carregado reproduz a perda de validação da época {melhor_epoca_q3} ({_perda_carregada:.4f}).")

**3d)** Em que época a perda de validação foi mínima? O que acontece com as duas curvas depois dela, e como se chama esse fenômeno? Por que guardar o estado com `copy.deepcopy`, em vez de apenas `modelo.state_dict()`?

**3e)** Compare a AUC da sua rede com a da regressão logística de referência e com o que você obteve nas listas 05 (TF-IDF) e 06 (boosting) para este mesmo alvo. O que os três resultados, juntos, sugerem sobre este problema?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 4 (2,0 pontos): caça aos erros

O código abaixo deveria treinar uma rede para separar as duas luas e medir a acurácia no teste. Ele tem **quatro** erros: um interrompe a execução e três são silenciosos. Corrija-os, mantendo a arquitetura (2 → 32 → 1), o otimizador e o número de passos.

In [ ]:
X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=7)
Xt_np, yt_np = make_moons(n_samples=1000, noise=0.2, random_state=8)
X_q4, Xt_q4 = torch.tensor(X_np, dtype=torch.float32), torch.tensor(Xt_np, dtype=torch.float32)

In [ ]:
# ✏️ Questão 4
torch.manual_seed(SEMENTE)
# SEU CÓDIGO AQUI
y_q4 = torch.tensor(y_np)
modelo_q4 = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())
otimizador_q4 = torch.optim.Adam(modelo_q4.parameters(), lr=0.01)
perda_fn = nn.BCEWithLogitsLoss()

for passo in range(800):
    perda_q4 = perda_fn(modelo_q4(X_q4).squeeze(1), y_q4)
    perda_q4.backward()
    otimizador_q4.step()

modelo_q4.eval()
with torch.no_grad():
    logits_teste = modelo_q4(Xt_q4)
acuracia_q4 = ((logits_teste > 0).numpy() == yt_np).mean()
print(f"perda final {perda_q4.item():.4f} · acurácia no teste {acuracia_q4:.3f}")

In [ ]:
# ✅ Conferência da questão 4 (não altere)
assert not any(isinstance(camada, nn.Sigmoid) for camada in modelo_q4.modules()), "ainda há uma Sigmoid dentro do modelo"
assert [tuple(p.shape) for p in modelo_q4.parameters()] == [(32, 2), (32,), (1, 32), (1,)], "mantenha a arquitetura 2 -> 32 -> 1"
with torch.no_grad():
    _logits = modelo_q4(Xt_q4).reshape(-1)
_acuracia = ((_logits > 0).numpy() == yt_np).mean()
assert abs(acuracia_q4 - _acuracia) < 1e-9, "a acurácia informada não corresponde à do modelo: confira as formas na comparação"
assert _acuracia > 0.95 and perda_q4.item() < 0.15, "o treino ainda não está funcionando como deveria"
print(f"Os quatro erros foram corrigidos: acurácia de {_acuracia:.1%} no teste.")

**4b)** Para cada um dos quatro erros, escreva: o que estava errado, qual era o sintoma (mensagem de erro, ou o que se observava no resultado) e como você o encontraria em um código que não soubesse estar errado.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 5 (1,0 ponto): PyTorch ou scikit-learn?

a) Dê um exemplo de problema em que você usaria o scikit-learn (ou o boosting da aula 06) e um em que usaria o PyTorch, justificando cada escolha em duas ou três frases.

b) Declare como usou assistentes de IA nesta lista: em quais questões, para quê, e o que precisou corrigir.

✏️ **Sua resposta:**

_Escreva aqui._